# Phase 3-4 - Uplift Models and Evaluation

Renders the results written by `python -m uplift.train` (`docs/uplift_results.json`). All numbers are on the held-out **test** split; the best model was chosen on validation.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

results = json.loads(Path('../docs/uplift_results.json').read_text())
best = results['best_model']
print('best model (chosen on validation):', best)

## Qini curves - all models, with the random-targeting line

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, curve in results['qini_curves'].items():
    ax.plot(curve['x'], curve['y'], label=name, linewidth=2 if name == best else 1)
ref = results['qini_curves'][best]
ax.plot([0, ref['x'][-1]], [0, ref['y'][-1]], 'k--', label='random line')
ax.set_xlabel('users targeted (ranked by predicted uplift)')
ax.set_ylabel('cumulative incremental outcomes (Qini)')
ax.legend()
plt.show()

## Qini AUC with bootstrap 95% CIs, and uplift@k

In [ ]:
table = pd.DataFrame(results['test_metrics']).T
table[['qini_auc', 'ci_low', 'ci_high', 'uplift_at_10', 'uplift_at_20', 'uplift_at_30']].sort_values(
    'qini_auc', ascending=False
)

## Observed uplift by predicted-uplift decile (best model)

Decile 1 = highest predicted uplift; observed uplift should fall.

In [ ]:
deciles = pd.DataFrame(results['decile_table'])
ax = deciles.plot.bar(x='decile', y='uplift', legend=False, figsize=(8, 4))
ax.set_ylabel('observed treated - control rate')
plt.show()

## Segments

Defined from model scores (individual response types are unobservable); `observed_uplift` is the sanity check.

In [ ]:
pd.DataFrame(results['segment_table'])